Python notebook to create/parse the datasets used for backbone swap training
(Pile-NER, CC related datasets) -- GLiNER (v1) equivalent.

GLiNER2 version this replaces: FINETUNES/GLINER/gliner2_custom_dataset.ipynb

Produces the same char-span intermediates as the GLiNER2 notebook, but
converts them to GLiNER v1's native training shape
    {"tokenized_text": [...], "ner": [[start, end_inclusive, label], ...]}
instead of GLiNER2's {"text": ..., "entities": {...}} shape, since gliner2
checkpoints cannot be loaded back via the `gliner` library.

Stage 1 (PileNER) reuses FINETUNES/GLINER/process_pilener.py's
extract_entity_spans() verbatim -- it already outputs GLiNER v1's format
directly, so unlike the GLiNER2 notebook, nothing needs rewriting here.

Stage 2 (IBMCCNER + BioDivNER + ClimateIE) reuses the exact same char-span
loaders as the GLiNER2 notebook, but converts via dataset_processing.py's
convert_to_token_spans() instead of char_spans_to_gliner2_examples().

## Data and Libraries

In [ ]:
# !pip install catalogue
# !pip install confection
# !pip install spacy==3.7.5
# !pip install --upgrade numpy
# !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz


In [1]:
import os
import json
import random
import ast

from collections import defaultdict

from datasets import load_dataset
import spacy
from tqdm import tqdm

from dataset_processing import (
    IBMCCNER_DIR, IBMCCNER_LABELS, ibmccner_process_bio_documents,
    BIODIVNER_DIR, BIODIVNER_LABELS, biodivner_process_bio_documents,
    convert_to_token_spans,
    tokenize_text,
)

c:\Users\Desktop\.conda\envs\clirener_finetune_gliner\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions

### Helpers Pile-CCNER

In [2]:
def load_ibmccner_char_spans(dataset_id=IBMCCNER_DIR, split=("train", "validation", "test")):
    print(f"[IBMCCNER] Loading '{dataset_id}' splits={split}")
    ds = load_dataset(dataset_id)
    train_documentwise = []
    temp_list = []
    for sp in split:
        for line in ds[sp]["text"]:
            if line.strip().startswith("-DOCSTART-"):
                if temp_list:
                    train_documentwise.append(temp_list)
                temp_list = []
            temp_list.append(line)
        if temp_list:
            train_documentwise.append(temp_list)
            temp_list = []

    print("[IBMCCNER] Converting BIO -> char spans")
    structured = ibmccner_process_bio_documents(
        document_list=train_documentwise,
        labels_to_keep=IBMCCNER_LABELS,
    )
    print(f"[IBMCCNER] {len(structured)} sentences")
    return structured


def load_biodivner_char_spans(data_dir=BIODIVNER_DIR, split=("train", "test", "dev")):
    print(f"[BioDivNER] Loading '{data_dir}' splits={split}")
    structured = biodivner_process_bio_documents(
        data_dir=data_dir,
        labels_to_keep=BIODIVNER_LABELS,
        split=list(split),
    )
    print(f"[BioDivNER] {len(structured)} sentences")
    return structured


def load_climateie_char_spans(data_dir="DATA/ClimateIE/human_corpus/", spacy_model="en_core_sci_sm"):
    """
    ClimateIE ships as whole-document JSONs with a flat `entities` dict
    keyed by opaque IDs; character offsets are GLOBAL (document-level).
    We sentence-split with spaCy and remap entity offsets to be
    sentence-local, keeping only entities fully contained in one sentence.
    Mirrors SCRIPT_VERSIONS/preprocess_other_ner_datasets.py exactly --
    identical to the GLiNER2 notebook's loader, unchanged.
    """
    print(f"[ClimateIE] Loading spaCy sentence splitter ({spacy_model})")
    nlp = spacy.load(spacy_model, disable=["ner"])

    try:
        file_names = [f for f in os.listdir(data_dir) if f.endswith(".json")]
    except FileNotFoundError:
        print(f"[ClimateIE] ERROR: directory not found: {data_dir}")
        return []

    structured = []
    for file_name in file_names:
        file_path = os.path.join(data_dir, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            document = json.load(f)

        raw_text = document["text"]
        raw_entities = document.get("entities", {})

        nlp.max_length = len(raw_text) + 100_000
        doc = nlp(raw_text)

        entity_list = list(raw_entities.values())
        entity_list.sort(key=lambda x: x["begin"])

        for sent in doc.sents:
            sent_text = sent.text
            sent_start, sent_end = sent.start_char, sent.end_char

            local_entities = []
            for ent in entity_list:
                # Keep only entities strictly contained within this sentence
                if ent["begin"] >= sent_start and ent["end"] <= sent_end:
                    local_entities.append({
                        "text": ent["substring"],
                        "label": ent["label"],
                        "start": ent["begin"] - sent_start,
                        "end": ent["end"] - sent_start,
                    })

            if sent_text.strip():
                structured.append({
                    "text": sent_text,
                    "entities": local_entities,
                })

    print(f"[ClimateIE] {len(structured)} sentences from {len(file_names)} documents")
    return structured

### Helpers Pile-NER

In [5]:
def load_raw_pilener(filepath):
    print(f"[PileNER] Loading raw file: {filepath}")
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"[PileNER] {len(data)} raw conversation entries")
    return data


def extract_entity_spans(entry):
    """Verbatim from FINETUNES/GLINER/process_pilener.py."""
    len_start = len("What describes ")
    len_end = len(" in the text?")
    entity_types, entity_texts, negative = [], [], []

    for c in entry['conversations']:
        if c['from'] == 'human' and c['value'].startswith('Text: '):
            text = c['value'][len('Text: '):]
            tokenized_text = tokenize_text(text)
        elif c['from'] == 'human' and c['value'].startswith('What describes '):
            entity_type = c['value'][len_start:-len_end]
            entity_types.append(entity_type)
        elif c['from'] == 'gpt' and c['value'].startswith('['):
            if c['value'] == '[]':
                negative.append(entity_types.pop())
                continue
            texts_ents = ast.literal_eval(c['value'])
            entity_texts.extend(texts_ents)
            num_repeat = len(texts_ents) - 1
            entity_types.extend([entity_types[-1]] * num_repeat)

    entity_spans = []
    for j, entity_text in enumerate(entity_texts):
        entity_tokens = tokenize_text(entity_text)
        matches = []
        for i in range(len(tokenized_text) - len(entity_tokens) + 1):
            if " ".join(tokenized_text[i:i + len(entity_tokens)]).lower() == " ".join(entity_tokens).lower():
                matches.append((i, i + len(entity_tokens) - 1, entity_types[j]))
        if matches:
            entity_spans.extend(matches)

    return {"tokenized_text": tokenized_text, "ner": entity_spans, "negative": negative}


def process_pilener(raw_data):
    """Verbatim from FINETUNES/GLINER/process_pilener.py's process_data()."""
    all_data = [extract_entity_spans(entry) for entry in tqdm(raw_data)]
    return all_data

# Main Code Pile-CCNER (gliner)

In [6]:
output_file = "pile_ccner_2025_gliner.json"
ibmccner_dataset = IBMCCNER_DIR
biodivner_dir = "C:\\RAD\\CLIRENER\\CliReNER_backbone_switch\\CliReNER\\DATA\\BiodivNER"
climateie_dir = "C:\\RAD\\CLIRENER\\CliReNER_backbone_switch\\CliReNER\\DATA\\ClimateIE\\human_corpus"
spacy_model = "en_core_sci_sm"

no_shuffle = False
seed = 301202

# =====================================================================
# Main Execution Cell
# =====================================================================
all_examples = []

# --- IBMCCNER ---
ibmccner_docs = load_ibmccner_char_spans(dataset_id=ibmccner_dataset)
print("[IBMCCNER] Converting to GLiNER v1 format (tokenized_text/ner)")
ibmccner_examples = convert_to_token_spans(ibmccner_docs)
for ex in ibmccner_examples:
    ex["source"] = "IBMCCNER"
all_examples.extend(ibmccner_examples)

# --- BioDivNER ---
biodivner_docs = load_biodivner_char_spans(data_dir=biodivner_dir)
print("[BioDivNER] Converting to GLiNER v1 format (tokenized_text/ner)")
biodivner_examples = convert_to_token_spans(biodivner_docs)
for ex in biodivner_examples:
    ex["source"] = "BioDivNER"
all_examples.extend(biodivner_examples)

# --- ClimateIE (human_corpus only) ---
climateie_docs = load_climateie_char_spans(data_dir=climateie_dir, spacy_model=spacy_model)
print("[ClimateIE] Converting to GLiNER v1 format (tokenized_text/ner)")
climateie_examples = convert_to_token_spans(climateie_docs)
for ex in climateie_examples:
    ex["source"] = "ClimateIE"
all_examples.extend(climateie_examples)

print(f"\nTotal combined examples: {len(all_examples)}")

# --- Shuffle for good source interleaving (matters for Stage-2 training) ---
if not no_shuffle:
    random.seed(seed)
    random.shuffle(all_examples)

# --- Write as a single JSON array. This matches process_pilener.py's own
#     save convention (json.dump, not JSONL) and is exactly what GLiNER v1's
#     Trainer expects as train_dataset/eval_dataset once json.load()-ed back
#     into a plain Python list -- see EXPERIMENTS/finetune.py:train_gliner(). ---
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_examples, f, ensure_ascii=False)

print(f"Saved: {os.path.abspath(output_file)}")

# --- Quick summary ---
label_counts = defaultdict(int)
source_counts = defaultdict(int)
source_label_counts = defaultdict(lambda: defaultdict(int))
zero_entity_counts = defaultdict(int)
for ex in all_examples:
    source_counts[ex["source"]] += 1
    if not ex["ner"]:
        zero_entity_counts[ex["source"]] += 1
    for start, end, label in ex["ner"]:
        label_counts[label] += 1
        source_label_counts[ex["source"]][label] += 1

print("\nPer-source example counts "
      "(incl. zero-entity sentences -- convert_to_token_spans() keeps everything by construction):")
for src, cnt in sorted(source_counts.items()):
    print(f"  {src:<12}: {cnt}  (zero-entity: {zero_entity_counts.get(src, 0)})")

print("\nPer-label entity counts (combined):")
for label, cnt in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:<30}: {cnt}")

print("\nPer-source label breakdown:")
for src in sorted(source_label_counts.keys()):
    print(f"  [{src}]")
    for label, cnt in sorted(source_label_counts[src].items(), key=lambda x: -x[1]):
        print(f"    {label:<28}: {cnt}")

[IBMCCNER] Loading 'ibm-research/Climate-Change-NER' splits=('train', 'validation', 'test')
[IBMCCNER] Converting BIO -> char spans
[IBMCCNER] 522 sentences
[IBMCCNER] Converting to GLiNER v1 format (tokenized_text/ner)
[BioDivNER] Loading 'C:\RAD\CLIRENER\CliReNER_backbone_switch\CliReNER\DATA\BiodivNER' splits=('train', 'test', 'dev')
Processing file: C:\RAD\CLIRENER\CliReNER_backbone_switch\CliReNER\DATA\BiodivNER\test.csv...
Processing file: C:\RAD\CLIRENER\CliReNER_backbone_switch\CliReNER\DATA\BiodivNER\train.csv...
[BioDivNER] 2158 sentences
[BioDivNER] Converting to GLiNER v1 format (tokenized_text/ner)
[ClimateIE] Loading spaCy sentence splitter (en_core_sci_sm)
[ClimateIE] 6552 sentences from 25 documents
[ClimateIE] Converting to GLiNER v1 format (tokenized_text/ner)

Total combined examples: 9232
Saved: c:\RAD\CLIRENER\CliReNER_backbone_switch\CliReNER\FINETUNES\GLINER\pile_ccner_2025_gliner.json

Per-source example counts (incl. zero-entity sentences -- convert_to_token_sp

# Main Code Pile-NER

In [7]:

INPUT_FILE = "C:\\RAD\\CLIRENER\\CliReNER_backbone_switch\\CliReNER\\DATA\\PileNER\\train.json"
OUTPUT_FILE = "pilener_2025_gliner.json"
MAX_EXAMPLES = None
SHUFFLE = True
SEED = 301202

# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------

raw_data = load_raw_pilener(INPUT_FILE)
examples = process_pilener(raw_data)
for ex in examples:
    ex["source"] = "PileNER"

if SHUFFLE:
    random.seed(SEED)
    random.shuffle(examples)

if MAX_EXAMPLES is not None:
    examples = examples[:MAX_EXAMPLES]
    print(f"[PileNER] Capped to {len(examples)} examples")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    try:
        json.dump(examples, f, ensure_ascii=False)
    except UnicodeEncodeError:
        json.dump(examples, f, ensure_ascii=True)

print(f"Saved: {OUTPUT_FILE}")

# --- Quick label distribution summary ---
label_counts = defaultdict(int)
negative_counts = defaultdict(int)
zero_entity = 0
for ex in examples:
    if not ex["ner"]:
        zero_entity += 1
    for start, end, label in ex["ner"]:
        label_counts[label] += 1
    for label in ex["negative"]:
        negative_counts[label] += 1

print(f"\nTotal examples: {len(examples)}")
print(f"Zero-entity examples (kept, ner=[]): {zero_entity}")
print(f"Unique entity types (positive somewhere): {len(label_counts)}")
print(f"Unique entity types (negative somewhere): {len(negative_counts)}")

print("\nTop 20 entity types by mention count:")
for label, cnt in sorted(label_counts.items(), key=lambda x: -x[1])[:20]:
    print(f"  {label:<30}: {cnt}")

[PileNER] Loading raw file: C:\RAD\CLIRENER\CliReNER_backbone_switch\CliReNER\DATA\PileNER\train.json
[PileNER] 45889 raw conversation entries


100%|██████████| 45889/45889 [00:39<00:00, 1174.68it/s]


Saved: pilener_2025_gliner.json

Total examples: 45889
Zero-entity examples (kept, ner=[]): 384
Unique entity types (positive somewhere): 15176
Unique entity types (negative somewhere): 9074

Top 20 entity types by mention count:
  concept                       : 43743
  Person                        : 40263
  person                        : 38936
  Organization                  : 38259
  Location                      : 32023
  organization                  : 31526
  product                       : 28823
  location                      : 28170
  variable                      : 21974
  Concept                       : 15370
  object                        : 15115
  Product                       : 12304
  technology                    : 11391
  Date                          : 10597
  chemical                      : 9881
  Medical Condition             : 9812
  software                      : 9655
  number                        : 9606
  medical condition             : 9477
  disease      